# Basic interaction with OpenAI GPT models

This example demonstrates how to set up an OpenAI GPT model and send a basic user prompt using a Jypiter notebook.

## 1. Setup

To use the OpenAI API, you need to have an API key:

1. Create or log into your [OpenAI account](https://platform.openai.com/)
2. Go to [API Keys](https://platform.openai.com/api-keys)
3. Click "Create new secret key", name it, copy it immediately
4. Add billing/payment details so the key becomes active (the API is pay-per-use)

Once you have the API key, you need to export its value as and environment variable called `OPENAI_API_KEY`. If this env is not defined, you will be asked for your API key (it will not be stored in the notebook).

In [1]:
!pip install openai

import os
import time
from getpass import getpass
from openai import OpenAI

# Ask for the API key interactively if not defined as env variable
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI()

Enter your OpenAI API key: ··········


## 2. Function to query the model

This section defines a helper function to send a user prompt to the model.

In [2]:
def query_model(prompt: str,
                model: str = "gpt-6-luna",
                max_tokens: int = 4096,
                temperature: float = 0,
                reasoning: str = "none") -> str:
    """Send a user prompt to an OpenAI model and return the text response."""
    params = {
        "model": model,
        "input": prompt,
        "max_output_tokens": max_tokens,
        "reasoning": {"effort": reasoning},
    }
    # Sampling controls such as temperature are accepted only without reasoning
    if reasoning == "none":
        params["temperature"] = temperature

    start = time.perf_counter()
    response = client.responses.create(**params)
    latency = time.perf_counter() - start

    # Log some details about the response
    usage = response.usage
    print(f"\tModel: {response.model}")
    print(f"\tLatency: {latency:.3f} seconds")
    print(f"\tInput tokens: {usage.input_tokens}")
    print(f"\tOutput tokens: {usage.output_tokens}")
    print(f"\tReasoning tokens: {usage.output_tokens_details.reasoning_tokens}")
    print(f"\tTotal tokens: {usage.total_tokens}")

    return response.output_text

## 3. Send user prompt

This section sends and user prompt to the model using the function previously defined.

In [3]:
prompt = "How many tokens are in your context window?"

print("=== Without reasoning ===")
print("User:", prompt)
response = query_model(prompt)
print("AI:", response)
print()
print("=== With reasoning ===")
print("User:", prompt)
response = query_model(prompt, reasoning="medium")
print("AI:", response)

=== Without reasoning ===
User: How many tokens are in your context window?
	Model: gpt-6-luna
	Latency: 4.531 seconds
	Input tokens: 15
	Output tokens: 22
	Reasoning tokens: 0
	Total tokens: 37
AI: I don’t have access to an exact token-count limit for this conversation’s context window.

=== With reasoning ===
User: How many tokens are in your context window?
	Model: gpt-6-luna
	Latency: 2.553 seconds
	Input tokens: 15
	Output tokens: 99
	Reasoning tokens: 60
	Total tokens: 114
AI: I don’t have access to the exact context-window limit for this chat. The context window includes the conversation and other material provided to me, not just my replies.
